In [1]:
# ============================================================
# OPENLENS
# 09. OPTIMIERTE RAG-SUCHE MIT MISTRAL 7B IN 4-BIT
# ============================================================
#
# Dieses Notebook:
# - erkennt den OpenLens-Projektordner automatisch
# - prüft CUDA und die GPU
# - öffnet die vorhandene ChromaDB
# - lädt das lokale Embedding-Modell
# - lädt Mistral 7B in 4-Bit
# - reduziert den RAG-Kontext für schnellere Antworten
# - streamt die Mistral-Antwort live in das Notebook
# - zeigt Quellen und Laufzeiten an
#
# Notebook:
# OpenLens/Datenbank/09 RAG search.ipynb
#
# Voraussetzungen:
# - ChromaDB wurde aufgebaut
# - Embedding-Modell wurde lokal gespeichert
# - Mistral liegt unter:
#   OpenLens/Modelle/Mistral-7B-Instruct-v0.3
# - CUDA-fähiges PyTorch ist installiert
#
# ============================================================


# ============================================================
# 1. IMPORTE UND PAKETE
# ============================================================

import gc
import importlib
import json
import queue
import re
import subprocess
import sys
import threading
import time
import warnings
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore")


BENOETIGTE_PAKETE = {
    "chromadb": "chromadb",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "numpy": "numpy",
    "pandas": "pandas",
}


def paket_installieren_falls_noetig(
    import_name: str,
    pip_name: str,
) -> None:
    """
    Installiert ein Paket nur dann, wenn es im aktuellen
    Notebook-Environment nicht importiert werden kann.
    """

    try:

        importlib.import_module(
            import_name
        )

    except ImportError:

        print(
            f"{pip_name} fehlt und wird installiert ..."
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                pip_name,
            ]
        )


for import_name, pip_name in BENOETIGTE_PAKETE.items():

    paket_installieren_falls_noetig(
        import_name=import_name,
        pip_name=pip_name,
    )


try:

    import torch

except ImportError as fehler:

    raise ImportError(
        "\nPyTorch wurde nicht gefunden.\n\n"
        "Führe zuerst dein Notebook für das "
        "PyTorch-CUDA-Setup aus."
    ) from fehler


import chromadb
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TextIteratorStreamer,
)


# ============================================================
# 2. EINSTELLUNGEN
# ============================================================

COLLECTION_NAME = "fragdenstaat_openlens"


# Nur drei Quellen verwenden.
#
# Weniger Quellen:
# - kürzerer Prompt
# - schnellere Verarbeitung
# - geringerer Speicherverbrauch
TOP_K = 3


# ChromaDB darf zunächst etwas mehr Treffer liefern.
# Danach werden Duplikate entfernt.
INITIAL_TOP_K = 10


# Maximale Zahl erzeugter Antwort-Tokens.
#
# 120 reicht für einen schnellen Funktionstest.
MAX_NEW_TOKENS = 120


# Maximale Länge des vollständigen Prompts.
#
# Die alte Version nutzte bis zu 5.000 Tokens.
# Für die Demo reichen zunächst 1.800.
MAX_INPUT_TOKENS = 1800


# Maximale Zeichen pro Quelle vor der Tokenisierung.
MAX_CHARACTERS_PER_SOURCE = 1400


# Mindestähnlichkeit eines Chunks.
MIN_SIMILARITY = 0.15


# Das Embedding-Modell läuft auf der CPU.
# Dadurch bleibt der GPU-Speicher für Mistral frei.
EMBEDDING_DEVICE = "cpu"


# Nur lokal vorhandene Modelle laden.
LOCAL_FILES_ONLY = True


# Generierung bleibt sachlich und reproduzierbar.
DO_SAMPLE = False

TEMPERATURE = 0.0

REPETITION_PENALTY = 1.06


# Wiederholte Wortgruppen unterdrücken.
NO_REPEAT_NGRAM_SIZE = 4


# Wartezeit auf das nächste generierte Textstück.
STREAM_TIMEOUT_SECONDS = 120


# Abbruchkriterium für nahezu identische Chunks.
DUPLIKAT_SCHLUESSEL_LAENGE = 900


# Standardfrage für den ersten Test.
TESTFRAGE = (
    "Welche Themen behandeln die vorhandenen Dokumente "
    "im Zusammenhang mit Behörden, Informationszugang "
    "und staatlichen Entscheidungen?"
)


# ============================================================
# 3. PROJEKTORDNER ERKENNEN
# ============================================================

ARBEITSORDNER = Path.cwd().resolve()


BEKANNTE_UNTERORDNER = {
    "datenbank",
    "dokumente",
    "texte",
    "bereinigte_texte",
    "chunks",
    "embeddings",
    "modelle",
    "chromadb",
    "rag_ergebnisse",
    "ocr_kandidaten",
}


if ARBEITSORDNER.name.lower() in BEKANNTE_UNTERORDNER:

    PROJEKTORDNER = ARBEITSORDNER.parent


elif (
    ARBEITSORDNER
    / "Datenbank"
).is_dir():

    PROJEKTORDNER = ARBEITSORDNER


else:

    FALLBACK_PROJEKTORDNER = Path(
        r"C:\Users\Admin\Desktop\OpenLens"
    )

    if FALLBACK_PROJEKTORDNER.is_dir():

        PROJEKTORDNER = FALLBACK_PROJEKTORDNER

    else:

        raise FileNotFoundError(
            "\nDer OpenLens-Projektordner konnte "
            "nicht erkannt werden.\n\n"
            f"Aktueller Arbeitsordner:\n"
            f"{ARBEITSORDNER}\n\n"
            "Öffne das Notebook aus dem OpenLens-Hauptordner "
            "oder aus OpenLens\\Datenbank."
        )


DATENBANKORDNER = (
    PROJEKTORDNER
    / "Datenbank"
)

CHROMA_PATH = (
    PROJEKTORDNER
    / "ChromaDB"
)

MODELLORDNER = (
    PROJEKTORDNER
    / "Modelle"
)

RAG_ERGEBNISORDNER = (
    PROJEKTORDNER
    / "RAG_Ergebnisse"
)


EMBEDDING_MODEL_PATH = (
    MODELLORDNER
    / "paraphrase-multilingual-MiniLM-L12-v2"
)

MISTRAL_PATH = (
    MODELLORDNER
    / "Mistral-7B-Instruct-v0.3"
)

CHROMA_MANIFEST_PATH = (
    CHROMA_PATH
    / "openlens_chromadb_manifest.json"
)


RAG_ERGEBNISORDNER.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. SYSTEM UND PFADE PRÜFEN
# ============================================================

print("=" * 80)
print("OPENLENS – OPTIMIERTE RAG-SUCHE")
print("=" * 80)

print("\nPython:")
print(sys.executable)

print("\nArbeitsordner:")
print(ARBEITSORDNER)

print("\nProjektordner:")
print(PROJEKTORDNER)

print("\nPyTorch-Version:")
print(torch.__version__)

print("\nPyTorch-CUDA-Version:")
print(torch.version.cuda)

print("\nCUDA verfügbar:")
print(torch.cuda.is_available())


if not torch.cuda.is_available():

    raise RuntimeError(
        "\nCUDA ist in PyTorch nicht verfügbar.\n\n"
        "Führe zuerst dein Notebook "
        "'08 PyTorch CUDA.ipynb' aus."
    )


GPU_NAME = torch.cuda.get_device_name(
    0
)

GPU_GESAMTSPEICHER_GB = (
    torch.cuda.get_device_properties(
        0
    ).total_memory
    / 1024**3
)


print("\nGPU:")
print(GPU_NAME)

print("\nGPU-Gesamtspeicher:")
print(f"{GPU_GESAMTSPEICHER_GB:.2f} GB")

print("\nChromaDB:")
print(CHROMA_PATH)

print("\nEmbedding-Modell:")
print(EMBEDDING_MODEL_PATH)

print("\nMistral-Modell:")
print(MISTRAL_PATH)


if not CHROMA_PATH.is_dir():

    raise FileNotFoundError(
        "\nDie ChromaDB wurde nicht gefunden:\n"
        f"{CHROMA_PATH}"
    )


if not EMBEDDING_MODEL_PATH.is_dir():

    raise FileNotFoundError(
        "\nDas Embedding-Modell wurde nicht gefunden:\n"
        f"{EMBEDDING_MODEL_PATH}"
    )


if not MISTRAL_PATH.is_dir():

    raise FileNotFoundError(
        "\nMistral wurde nicht gefunden:\n"
        f"{MISTRAL_PATH}"
    )


print("\nAlle benötigten Ordner wurden gefunden.")


# ============================================================
# 5. CHROMADB ÖFFNEN
# ============================================================

print("\n" + "=" * 80)
print("CHROMADB WIRD GEÖFFNET")
print("=" * 80)


chroma_client = chromadb.PersistentClient(
    path=str(
        CHROMA_PATH
    )
)


vorhandene_collections = []

for collection_eintrag in chroma_client.list_collections():

    collection_name = getattr(
        collection_eintrag,
        "name",
        None,
    )

    if collection_name is None:

        collection_name = str(
            collection_eintrag
        )

    vorhandene_collections.append(
        collection_name
    )


print("\nGefundene Collections:")

for name in vorhandene_collections:

    print("-", name)


if COLLECTION_NAME not in vorhandene_collections:

    raise RuntimeError(
        "\nDie erwartete Collection wurde nicht gefunden.\n\n"
        f"Erwartet: {COLLECTION_NAME}\n"
        f"Vorhanden: {vorhandene_collections}"
    )


collection = chroma_client.get_collection(
    name=COLLECTION_NAME
)


GESPEICHERTE_CHUNKS = collection.count()


if GESPEICHERTE_CHUNKS <= 0:

    raise RuntimeError(
        "\nDie Collection enthält keine Chunks."
    )


print("\nVerwendete Collection:")
print(collection.name)

print("\nGespeicherte Chunks:")
print(f"{GESPEICHERTE_CHUNKS:,}")


# ============================================================
# 6. CHROMADB-MANIFEST LADEN
# ============================================================

chroma_manifest = {}


if CHROMA_MANIFEST_PATH.exists():

    with CHROMA_MANIFEST_PATH.open(
        "r",
        encoding="utf-8",
    ) as datei:

        chroma_manifest = json.load(
            datei
        )

    print("\nChromaDB-Manifest wurde geladen.")

else:

    print(
        "\nHinweis: Kein ChromaDB-Manifest gefunden. "
        "Die Dimension wird direkt geprüft."
    )


# ============================================================
# 7. EMBEDDING-MODELL LADEN
# ============================================================

print("\n" + "=" * 80)
print("EMBEDDING-MODELL WIRD GELADEN")
print("=" * 80)


embedding_model = SentenceTransformer(
    str(
        EMBEDDING_MODEL_PATH
    ),
    device=EMBEDDING_DEVICE,
    local_files_only=LOCAL_FILES_ONLY,
)


EMBEDDING_DIMENSION = (
    embedding_model
    .get_sentence_embedding_dimension()
)


print("\nEmbedding-Modell geladen.")

print("\nGerät:")
print(EMBEDDING_DEVICE)

print("\nEmbedding-Dimension:")
print(EMBEDDING_DIMENSION)


# ============================================================
# 8. EMBEDDING-DIMENSION PRÜFEN
# ============================================================

embedding_beispiel = collection.get(
    limit=1,
    include=[
        "embeddings",
    ],
)


gespeicherte_embeddings = (
    embedding_beispiel.get(
        "embeddings"
    )
)


if (
    gespeicherte_embeddings is None
    or len(
        gespeicherte_embeddings
    ) == 0
):

    raise RuntimeError(
        "\nIn ChromaDB wurden keine Embeddings gefunden."
    )


CHROMA_EMBEDDING_DIMENSION = len(
    gespeicherte_embeddings[0]
)


print("\nChromaDB-Embedding-Dimension:")
print(CHROMA_EMBEDDING_DIMENSION)


if (
    EMBEDDING_DIMENSION
    != CHROMA_EMBEDDING_DIMENSION
):

    raise ValueError(
        "\nDas Embedding-Modell passt nicht zur ChromaDB.\n\n"
        f"Embedding-Modell: {EMBEDDING_DIMENSION}\n"
        f"ChromaDB: {CHROMA_EMBEDDING_DIMENSION}"
    )


print("\nEmbedding-Modell und ChromaDB sind kompatibel.")


# ============================================================
# 9. GPU VOR DEM LADEN BEREINIGEN
# ============================================================

gc.collect()

torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()


# ============================================================
# 10. MISTRAL IN 4-BIT LADEN
# ============================================================

print("\n" + "=" * 80)
print("MISTRAL 7B WIRD IN 4-BIT GELADEN")
print("=" * 80)


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


print("\nTokenizer wird geladen ...")


tokenizer = AutoTokenizer.from_pretrained(
    str(
        MISTRAL_PATH
    ),
    local_files_only=LOCAL_FILES_ONLY,
    use_fast=True,
)


if tokenizer.pad_token_id is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )


print("Tokenizer wurde geladen.")

print("\nMistral wird quantisiert auf die GPU geladen ...")


try:

    model = AutoModelForCausalLM.from_pretrained(
        str(
            MISTRAL_PATH
        ),
        quantization_config=quantization_config,
        device_map={
            "": 0,
        },
        local_files_only=LOCAL_FILES_ONLY,
        low_cpu_mem_usage=True,
        dtype=torch.float16,
    )

except TypeError:

    # Kompatibilität mit älteren Transformers-Versionen.
    model = AutoModelForCausalLM.from_pretrained(
        str(
            MISTRAL_PATH
        ),
        quantization_config=quantization_config,
        device_map={
            "": 0,
        },
        local_files_only=LOCAL_FILES_ONLY,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )

except Exception as fehler:

    raise RuntimeError(
        "\nMistral konnte nicht in 4-Bit geladen werden.\n\n"
        "Mögliche Ursachen:\n"
        "- bitsandbytes funktioniert unter deiner "
        "Installation nicht korrekt,\n"
        "- zu wenig GPU-Speicher ist frei,\n"
        "- ein anderer Kernel belegt die GPU,\n"
        "- die CUDA- und PyTorch-Versionen passen nicht.\n\n"
        f"Originalfehler:\n{fehler}"
    ) from fehler


model.eval()


MODELL_GERAET = (
    model
    .get_input_embeddings()
    .weight
    .device
)


GPU_BELEGT_GB = (
    torch.cuda.memory_allocated(
        0
    )
    / 1024**3
)

GPU_RESERVIERT_GB = (
    torch.cuda.memory_reserved(
        0
    )
    / 1024**3
)


print("\nMistral wurde erfolgreich geladen.")

print("\nEingabegerät:")
print(MODELL_GERAET)

print("\nGPU-Speicher belegt:")
print(f"{GPU_BELEGT_GB:.2f} GB")

print("\nGPU-Speicher reserviert:")
print(f"{GPU_RESERVIERT_GB:.2f} GB")


if MODELL_GERAET.type != "cuda":

    raise RuntimeError(
        "\nMistral liegt nicht auf der CUDA-GPU.\n\n"
        f"Erkanntes Gerät: {MODELL_GERAET}"
    )


# ============================================================
# 11. ALLGEMEINE HILFSFUNKTIONEN
# ============================================================

def sicherer_textwert(
    wert: Any,
) -> str:
    """
    Wandelt beliebige Werte robust in Text um.
    """

    if wert is None:

        return ""


    try:

        if pd.isna(
            wert
        ):

            return ""

    except Exception:

        pass


    return str(
        wert
    ).strip()


def normalisiere_text(
    text: str,
) -> str:
    """
    Normalisiert Text zur Duplikaterkennung.
    """

    text = sicherer_textwert(
        text
    ).lower()


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()


def erster_vorhandener_wert(
    metadaten: dict[str, Any] | None,
    feldnamen: list[str],
    standardwert: Any = "",
) -> Any:
    """
    Gibt den ersten vorhandenen und nicht leeren
    Metadatenwert zurück.
    """

    if not metadaten:

        return standardwert


    for feldname in feldnamen:

        wert = metadaten.get(
            feldname
        )

        if wert not in (
            None,
            "",
            "None",
            "nan",
        ):

            return wert


    return standardwert


def standardisiere_quelle(
    chunk_id: str,
    text: str,
    metadaten: dict[str, Any] | None,
    distanz: float,
    rang: int,
) -> dict[str, Any]:
    """
    Vereinheitlicht die ChromaDB-Quellenmetadaten.
    """

    metadaten = metadaten or {}


    titel = erster_vorhandener_wert(
        metadaten,
        [
            "title",
            "document_title",
            "foirequest",
            "publicbody",
            "pdf_name",
        ],
        "Unbekanntes Dokument",
    )


    pdf_name = erster_vorhandener_wert(
        metadaten,
        [
            "pdf_name",
            "filename",
            "file_name",
        ],
        "",
    )


    seite_start = erster_vorhandener_wert(
        metadaten,
        [
            "page_start",
            "page",
            "page_number",
        ],
        "",
    )


    seite_ende = erster_vorhandener_wert(
        metadaten,
        [
            "page_end",
            "page",
            "page_number",
        ],
        "",
    )


    site_url = erster_vorhandener_wert(
        metadaten,
        [
            "site_url",
            "url",
            "source_url",
        ],
        "",
    )


    file_url = erster_vorhandener_wert(
        metadaten,
        [
            "file_url",
            "document_url",
            "download_url",
        ],
        "",
    )


    document_id = erster_vorhandener_wert(
        metadaten,
        [
            "document_id",
            "uid",
        ],
        "",
    )


    similarity = max(
        0.0,
        min(
            1.0,
            1.0 - float(
                distanz
            ),
        ),
    )


    return {
        "rank": rang,
        "chunk_id": str(
            chunk_id
        ),
        "document_id": document_id,
        "title": str(
            titel
        ),
        "pdf_name": str(
            pdf_name
        ),
        "page_start": seite_start,
        "page_end": seite_ende,
        "site_url": str(
            site_url
        ),
        "file_url": str(
            file_url
        ),
        "text": sicherer_textwert(
            text
        ),
        "distance": float(
            distanz
        ),
        "similarity": similarity,
        "metadata": metadaten,
    }


# ============================================================
# 12. SEMANTISCHE SUCHE
# ============================================================

def suche_relevante_chunks(
    frage: str,
    top_k: int = TOP_K,
) -> list[dict[str, Any]]:
    """
    Sucht relevante Chunks und entfernt weitgehend
    doppelte Treffer.
    """

    frage = sicherer_textwert(
        frage
    )


    if not frage:

        raise ValueError(
            "Die Frage darf nicht leer sein."
        )


    frage_embedding = embedding_model.encode(
        [
            frage,
        ],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )


    frage_embedding = np.asarray(
        frage_embedding,
        dtype=np.float32,
    )


    tatsaechliches_initial_top_k = min(
        INITIAL_TOP_K,
        GESPEICHERTE_CHUNKS,
    )


    suchergebnis = collection.query(
        query_embeddings=frage_embedding.tolist(),
        n_results=tatsaechliches_initial_top_k,
        include=[
            "documents",
            "metadatas",
            "distances",
        ],
    )


    ids = suchergebnis.get(
        "ids",
        [[]],
    )[0]

    dokumente = suchergebnis.get(
        "documents",
        [[]],
    )[0]

    metadaten_liste = suchergebnis.get(
        "metadatas",
        [[]],
    )[0]

    distanzen = suchergebnis.get(
        "distances",
        [[]],
    )[0]


    quellen = []

    gesehene_textteile = set()


    for rang, (
        chunk_id,
        text,
        metadaten,
        distanz,
    ) in enumerate(
        zip(
            ids,
            dokumente,
            metadaten_liste,
            distanzen,
        ),
        start=1,
    ):

        quelle = standardisiere_quelle(
            chunk_id=chunk_id,
            text=text,
            metadaten=metadaten,
            distanz=distanz,
            rang=len(
                quellen
            ) + 1,
        )


        if (
            quelle["similarity"]
            < MIN_SIMILARITY
        ):

            continue


        text_schluessel = normalisiere_text(
            quelle["text"]
        )[
            :DUPLIKAT_SCHLUESSEL_LAENGE
        ]


        if text_schluessel in gesehene_textteile:

            continue


        gesehene_textteile.add(
            text_schluessel
        )


        quellen.append(
            quelle
        )


        if len(
            quellen
        ) >= int(
            top_k
        ):

            break


    return quellen


# ============================================================
# 13. QUELLEN ANZEIGEN
# ============================================================

def zeige_quellen(
    quellen: list[dict[str, Any]],
    maximale_textlaenge: int = 700,
) -> None:
    """
    Zeigt die gefundenen Quellen kompakt an.
    """

    print("\n" + "=" * 80)
    print("GEFUNDENE QUELLEN")
    print("=" * 80)


    if not quellen:

        print(
            "\nEs wurden keine ausreichend ähnlichen "
            "Dokumentstellen gefunden."
        )

        return


    for quelle in quellen:

        print(
            f"\nQuelle {quelle['rank']}"
        )

        print("Titel:")
        print(
            quelle["title"]
        )


        if quelle["document_id"] != "":

            print("\nDokument-ID:")
            print(
                quelle["document_id"]
            )


        if quelle["page_start"] != "":

            if (
                quelle["page_end"] != ""
                and quelle["page_end"]
                != quelle["page_start"]
            ):

                print(
                    "\nSeiten:",
                    f"{quelle['page_start']}–"
                    f"{quelle['page_end']}",
                )

            else:

                print(
                    "\nSeite:",
                    quelle["page_start"],
                )


        print("\nÄhnlichkeit:")

        print(
            f"{quelle['similarity']:.4f}"
        )


        text = quelle["text"]


        if len(
            text
        ) > maximale_textlaenge:

            text = (
                text[
                    :maximale_textlaenge
                ]
                + " ..."
            )


        print("\nTextstelle:")
        print(text)

        print("-" * 80)


# ============================================================
# 14. KOMPAKTEN QUELLENKONTEXT ERSTELLEN
# ============================================================

def erstelle_quellenkontext(
    quellen: list[dict[str, Any]],
) -> str:
    """
    Baut einen kurzen, nummerierten Quellenkontext.
    """

    kontext_bloecke = []


    for quelle in quellen:

        text = quelle["text"][
            :MAX_CHARACTERS_PER_SOURCE
        ].strip()


        seitentext = ""


        if quelle["page_start"] != "":

            if (
                quelle["page_end"] != ""
                and quelle["page_end"]
                != quelle["page_start"]
            ):

                seitentext = (
                    f"Seiten "
                    f"{quelle['page_start']}–"
                    f"{quelle['page_end']}"
                )

            else:

                seitentext = (
                    f"Seite "
                    f"{quelle['page_start']}"
                )


        block_zeilen = [
            f"[QUELLE {quelle['rank']}]",
            f"Titel: {quelle['title']}",
        ]


        if seitentext:

            block_zeilen.append(
                f"Fundstelle: {seitentext}"
            )


        block_zeilen.extend(
            [
                "Text:",
                text,
                f"[/QUELLE {quelle['rank']}]",
            ]
        )


        kontext_bloecke.append(
            "\n".join(
                block_zeilen
            )
        )


    return "\n\n".join(
        kontext_bloecke
    )


# ============================================================
# 15. PROMPT ERSTELLEN
# ============================================================

SYSTEM_PROMPT = """
Du bist OpenLens, ein lokaler Assistent zur Analyse deutscher
Behörden- und Verwaltungsdokumente.

Verbindliche Regeln:

1. Verwende ausschließlich die bereitgestellten Quellen.
2. Erfinde keine Namen, Zahlen, Daten oder Zusammenhänge.
3. Belege wesentliche Aussagen mit [Quelle 1], [Quelle 2] usw.
4. Wenn die Quellen nicht ausreichen, sage das deutlich.
5. Antworte auf Deutsch.
6. Antworte kompakt und sachlich.
7. Wiederhole die vollständigen Quellentexte nicht.
""".strip()


def erstelle_formatierte_modelleingabe(
    frage: str,
    quellen: list[dict[str, Any]],
) -> dict[str, torch.Tensor]:
    """
    Erstellt den tokenisierten Mistral-Prompt.
    """

    quellenkontext = erstelle_quellenkontext(
        quellen
    )


    # Der Systemtext wird in die User-Nachricht integriert.
    # Dadurch bleibt die Eingabe mit unterschiedlichen
    # Mistral-Chat-Templates kompatibel.
    gesamter_prompt = f"""
ANWEISUNGEN:
{SYSTEM_PROMPT}

FRAGE:
{frage}

QUELLEN:
{quellenkontext}

AUFGABE:
Beantworte die Frage direkt anhand der Quellen.
Verwende Quellenverweise wie [Quelle 1].
Falls keine verlässliche Antwort möglich ist, schreibe:
"Die vorhandenen Quellen reichen für eine verlässliche Antwort nicht aus."
""".strip()


    nachrichten = [
        {
            "role": "user",
            "content": gesamter_prompt,
        }
    ]


    formatierter_prompt = tokenizer.apply_chat_template(
        nachrichten,
        tokenize=False,
        add_generation_prompt=True,
    )


    model_inputs = tokenizer(
        formatierter_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=False,
    )


    model_inputs = {
        schluessel: wert.to(
            MODELL_GERAET
        )
        for schluessel, wert
        in model_inputs.items()
    }


    return model_inputs


# ============================================================
# 16. LIVE-STREAMING-GENERIERUNG
# ============================================================

def erzeuge_rag_antwort_streaming(
    frage: str,
    quellen: list[dict[str, Any]],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> tuple[str, dict[str, Any]]:
    """
    Generiert die Antwort in einem Hintergrund-Thread und
    zeigt Textstücke bereits während der Erzeugung an.
    """

    if not quellen:

        antwort = (
            "Die vorhandenen Quellen reichen für eine "
            "verlässliche Antwort nicht aus."
        )

        print("\n" + antwort)

        return (
            antwort,
            {
                "generation_seconds": 0.0,
                "generated_tokens": 0,
                "tokens_per_second": 0.0,
                "input_tokens": 0,
            },
        )


    model_inputs = erstelle_formatierte_modelleingabe(
        frage=frage,
        quellen=quellen,
    )


    eingabe_token_anzahl = int(
        model_inputs[
            "input_ids"
        ].shape[1]
    )


    streamer = TextIteratorStreamer(
        tokenizer=tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
        timeout=STREAM_TIMEOUT_SECONDS,
        clean_up_tokenization_spaces=True,
    )


    generation_parameter = {
        **model_inputs,
        "streamer": streamer,
        "max_new_tokens": int(
            max_new_tokens
        ),
        "do_sample": DO_SAMPLE,
        "repetition_penalty": float(
            REPETITION_PENALTY
        ),
        "no_repeat_ngram_size": int(
            NO_REPEAT_NGRAM_SIZE
        ),
        "use_cache": True,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }


    if DO_SAMPLE:

        generation_parameter.update(
            {
                "temperature": max(
                    float(
                        TEMPERATURE
                    ),
                    0.01,
                ),
                "top_p": 0.9,
            }
        )


    thread_status = {
        "exception": None,
    }


    def generierungs_worker() -> None:
        """
        Führt model.generate() im Hintergrund aus.
        """

        try:

            with torch.inference_mode():

                model.generate(
                    **generation_parameter
                )

        except Exception as fehler:

            thread_status[
                "exception"
            ] = fehler

            # Den Stream trotzdem sauber beenden.
            try:

                streamer.on_finalized_text(
                    "",
                    stream_end=True,
                )

            except Exception:

                pass


    worker_thread = threading.Thread(
        target=generierungs_worker,
        daemon=True,
    )


    print("\n" + "=" * 80)
    print("LIVE-ANTWORT")
    print("=" * 80)
    print()


    startzeit = time.perf_counter()

    worker_thread.start()


    antwort_teile = []


    try:

        for text_stueck in streamer:

            print(
                text_stueck,
                end="",
                flush=True,
            )

            antwort_teile.append(
                text_stueck
            )

    except queue.Empty:

        raise TimeoutError(
            "\nMistral hat innerhalb von "
            f"{STREAM_TIMEOUT_SECONDS} Sekunden kein "
            "weiteres Textstück geliefert."
        )


    worker_thread.join(
        timeout=10
    )


    if thread_status["exception"] is not None:

        raise RuntimeError(
            "\nBei der Mistral-Generierung ist ein Fehler "
            "aufgetreten:\n"
            f"{thread_status['exception']}"
        ) from thread_status["exception"]


    generierungsdauer = (
        time.perf_counter()
        - startzeit
    )


    antwort = "".join(
        antwort_teile
    ).strip()


    print()


    if not antwort:

        raise ValueError(
            "\nMistral hat eine leere Antwort erzeugt."
        )


    antwort_token_ids = tokenizer.encode(
        antwort,
        add_special_tokens=False,
    )


    generierte_token_anzahl = len(
        antwort_token_ids
    )


    tokens_pro_sekunde = (
        generierte_token_anzahl
        / generierungsdauer
        if generierungsdauer > 0
        else 0.0
    )


    statistik = {
        "generation_seconds": round(
            generierungsdauer,
            3,
        ),
        "generated_tokens": int(
            generierte_token_anzahl
        ),
        "tokens_per_second": round(
            tokens_pro_sekunde,
            3,
        ),
        "input_tokens": int(
            eingabe_token_anzahl
        ),
    }


    return (
        antwort,
        statistik,
    )


# ============================================================
# 17. VOLLSTÄNDIGE RAG-PIPELINE
# ============================================================

def rag_search(
    frage: str,
    top_k: int = TOP_K,
    quellen_anzeigen: bool = True,
) -> dict[str, Any]:
    """
    Führt die vollständige OpenLens-RAG-Suche aus.
    """

    frage = sicherer_textwert(
        frage
    )


    if not frage:

        raise ValueError(
            "Bitte eine Frage eingeben."
        )


    gesamt_startzeit = time.perf_counter()


    print("\n" + "=" * 80)
    print("OPENLENS – RAG SEARCH")
    print("=" * 80)

    print("\nFrage:")
    print(frage)


    print(
        "\nRelevante Dokumentstellen werden gesucht ..."
    )


    such_startzeit = time.perf_counter()


    quellen = suche_relevante_chunks(
        frage=frage,
        top_k=top_k,
    )


    suchdauer = (
        time.perf_counter()
        - such_startzeit
    )


    print("\nGefundene verwertbare Quellen:")
    print(len(quellen))

    print("\nSuchdauer:")
    print(f"{suchdauer:.3f} Sekunden")


    if quellen_anzeigen:

        zeige_quellen(
            quellen
        )


    print(
        "\nMistral startet die Antwortgenerierung ..."
    )


    try:

        antwort, generierungsstatistik = (
            erzeuge_rag_antwort_streaming(
                frage=frage,
                quellen=quellen,
            )
        )

    except torch.cuda.OutOfMemoryError as fehler:

        torch.cuda.empty_cache()

        raise RuntimeError(
            "\nDer GPU-Speicher reicht für diese Anfrage "
            "nicht aus.\n\n"
            "Reduziere testweise:\n"
            "- TOP_K auf 2\n"
            "- MAX_INPUT_TOKENS auf 1200\n"
            "- MAX_NEW_TOKENS auf 80"
        ) from fehler


    gesamtdauer = (
        time.perf_counter()
        - gesamt_startzeit
    )


    ergebnis = {
        "question": frage,
        "answer": antwort,
        "sources": quellen,
        "source_count": len(
            quellen
        ),
        "retrieval_seconds": round(
            suchdauer,
            3,
        ),
        "input_tokens": (
            generierungsstatistik[
                "input_tokens"
            ]
        ),
        "generation_seconds": (
            generierungsstatistik[
                "generation_seconds"
            ]
        ),
        "generated_tokens": (
            generierungsstatistik[
                "generated_tokens"
            ]
        ),
        "tokens_per_second": (
            generierungsstatistik[
                "tokens_per_second"
            ]
        ),
        "total_seconds": round(
            gesamtdauer,
            3,
        ),
    }


    print("\n" + "=" * 80)
    print("LAUFZEIT")
    print("=" * 80)

    print("\nRetrieval:")
    print(
        f"{ergebnis['retrieval_seconds']:.3f} Sekunden"
    )

    print("\nEingabetokens:")
    print(
        ergebnis["input_tokens"]
    )

    print("\nGenerierung:")
    print(
        f"{ergebnis['generation_seconds']:.3f} Sekunden"
    )

    print("\nErzeugte Tokens:")
    print(
        ergebnis["generated_tokens"]
    )

    print("\nGeschwindigkeit:")
    print(
        f"{ergebnis['tokens_per_second']:.2f} "
        "Tokens/Sekunde"
    )

    print("\nGesamtdauer:")
    print(
        f"{ergebnis['total_seconds']:.3f} Sekunden"
    )


    return ergebnis


# ============================================================
# 18. ERGEBNIS SPEICHERN
# ============================================================

def speichere_rag_ergebnis(
    ergebnis: dict[str, Any],
) -> Path:
    """
    Speichert das letzte Ergebnis als JSON.
    """

    zeitstempel = time.strftime(
        "%Y-%m-%d_%H-%M-%S"
    )


    ausgabedatei = (
        RAG_ERGEBNISORDNER
        / f"rag_mistral_{zeitstempel}.json"
    )


    with ausgabedatei.open(
        "w",
        encoding="utf-8",
    ) as datei:

        json.dump(
            ergebnis,
            datei,
            ensure_ascii=False,
            indent=2,
            default=str,
        )


    print("\nErgebnis gespeichert unter:")
    print(ausgabedatei)


    return ausgabedatei


# ============================================================
# 19. TESTFRAGE AUSFÜHREN
# ============================================================

result = rag_search(
    frage=TESTFRAGE,
    top_k=TOP_K,
    quellen_anzeigen=True,
)


speichere_rag_ergebnis(
    result
)


# ============================================================
# 20. WEITERE FRAGEN
# ============================================================
#
# Für eine neue Frage in einer neuen Zelle:
#
# result = rag_search(
#     frage=(
#         "Welche Behörden werden in den "
#         "Dokumenten genannt?"
#     ),
#     top_k=3,
#     quellen_anzeigen=True,
# )
#
# speichere_rag_ergebnis(result)
#
# Die Antwort steht anschließend unter:
#
# result["answer"]
#
# Die Quellen stehen unter:
#
# result["sources"]
#
# ============================================================

W0724 21:46:21.125000 20148 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


OPENLENS – OPTIMIERTE RAG-SUCHE

Python:
c:\Users\Admin\Desktop\NEUE FISCHE\PANDAS-NUMPY-main\.venv\Scripts\python.exe

Arbeitsordner:
C:\Users\Admin\Desktop\OpenLens\Datenbank

Projektordner:
C:\Users\Admin\Desktop\OpenLens

PyTorch-Version:
2.11.0+cu128

PyTorch-CUDA-Version:
12.8

CUDA verfügbar:
True

GPU:
NVIDIA GeForce RTX 3060 Ti

GPU-Gesamtspeicher:
8.00 GB

ChromaDB:
C:\Users\Admin\Desktop\OpenLens\ChromaDB

Embedding-Modell:
C:\Users\Admin\Desktop\OpenLens\Modelle\paraphrase-multilingual-MiniLM-L12-v2

Mistral-Modell:
C:\Users\Admin\Desktop\OpenLens\Modelle\Mistral-7B-Instruct-v0.3

Alle benötigten Ordner wurden gefunden.

CHROMADB WIRD GEÖFFNET

Gefundene Collections:
- fragdenstaat_openlens

Verwendete Collection:
fragdenstaat_openlens

Gespeicherte Chunks:
271

ChromaDB-Manifest wurde geladen.

EMBEDDING-MODELL WIRD GELADEN


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Embedding-Modell geladen.

Gerät:
cpu

Embedding-Dimension:
384

ChromaDB-Embedding-Dimension:
384

Embedding-Modell und ChromaDB sind kompatibel.

MISTRAL 7B WIRD IN 4-BIT GELADEN

Tokenizer wird geladen ...
Tokenizer wurde geladen.

Mistral wird quantisiert auf die GPU geladen ...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


Mistral wurde erfolgreich geladen.

Eingabegerät:
cuda:0

GPU-Speicher belegt:
3.86 GB

GPU-Speicher reserviert:
3.99 GB

OPENLENS – RAG SEARCH

Frage:
Welche Themen behandeln die vorhandenen Dokumente im Zusammenhang mit Behörden, Informationszugang und staatlichen Entscheidungen?

Relevante Dokumentstellen werden gesucht ...

Gefundene verwertbare Quellen:
3

Suchdauer:
2.074 Sekunden

GEFUNDENE QUELLEN

Quelle 1
Titel:
Anregungen des Landesrechnungshofs zur Anpassung der Gesellschaftsverträge kommunaler Unternehmen

Dokument-ID:
171322

Seite: 1

Ähnlichkeit:
0.5920

Textstelle:
Landtag Brandenburg
Drucksache 3/499
3. Wahlperiode
Antwort
der Landesregierung
auf die Kleine Anfrage Nr. 181
des Abgeordneten Ralf Christoffers
Fraktion der PDS
Drucksache 3/364
Anregungen
egungen des Landesrechnungshofes zur Anpassung der Gesellschaftsverträge kommunaler Unternehmen
schaftsverträge kommunaler Unternehmen
Wortlaut der Kleinen Anfrage Nr. 181 vom 17.12.1999:
Der Landesrechnungshof hat in s

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


uständig. Die Landesbehörden sind nur für die
Ausführung der Vorschriften des Asylgesetzes zuständige
Stellen.
[/Quelle 3] (Quelle 4 fehlt)

ANSWER:
Die vorhandenen Behörden-Dokumente behandeln Themen im Zusammen-hang mit Behören, Informionszugang sowie staatlichen Entschiedungen. Hierbei geht es um die Anpassung von Gesellschafts-vertr

LAUFZEIT

Retrieval:
2.074 Sekunden

Eingabetokens:
1800

Generierung:
19.367 Sekunden

Erzeugte Tokens:
120

Geschwindigkeit:
6.20 Tokens/Sekunde

Gesamtdauer:
21.809 Sekunden

Ergebnis gespeichert unter:
C:\Users\Admin\Desktop\OpenLens\RAG_Ergebnisse\rag_mistral_2026-07-24_21-47-52.json


WindowsPath('C:/Users/Admin/Desktop/OpenLens/RAG_Ergebnisse/rag_mistral_2026-07-24_21-47-52.json')